# ASSIGNMENT CDS

# Problem Statement

Study analysis on the public sentiment towards a company and its stock effect their prices by extracting opinions from Twitter in determining whether public opinion has impact on the stock prices of the company.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sys
from collections import Counter

# Load dataset
df = pd.read_csv("dataset.csv")

print(f"Dataset loaded successfully!")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
df.head()

# Exploratory Data Analysis (EDA)
- Column names
- Missing values
- Duplicate rows
- Data types
- Date range
- Number of tweets per company/stock
- Number of tweets per date

## General Dataset Overview

In [ ]:
# --- 1. Column names and data types ---
print("=" * 50)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 50)
print(df.dtypes)
print()

In [ ]:
# --- 2. Shape of the dataset (rows, columns) ---
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print()

In [ ]:
# --- 3. Missing values per column ---
print("=" * 50)
print("MISSING VALUES PER COLUMN")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum():,}")

In [ ]:
# --- 4. Duplicate rows ---
print("=" * 50)
print("DUPLICATE ROWS")
print("=" * 50)
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count:,}")
if duplicate_count > 0:
    print("Removing duplicates...")
    df = df.drop_duplicates()
    print(f"Rows after removing duplicates: {len(df):,}")

In [ ]:
# --- 5. Basic descriptive statistics for numerical columns ---
print("=" * 50)
print("DESCRIPTIVE STATISTICS (NUMERICAL COLUMNS)")
print("=" * 50)
df.describe()

## Stock Exploratory Data Analysis

In [ ]:
# --- Number of tweets per company/stock ---
print("=" * 50)
print("NUMBER OF TWEETS PER STOCK/COMPANY")
print("=" * 50)
stock_counts = df['STOCK'].value_counts()
print(stock_counts)
print(f"\nTotal unique stocks/companies: {df['STOCK'].nunique()}")

In [ ]:
# --- Bar chart: Tweets per stock ---
plt.figure(figsize=(14, 6))
stock_counts = df['STOCK'].value_counts().head(20)
sns.barplot(x=stock_counts.index, y=stock_counts.values, palette='Blues_r')
plt.title('Top 20 Companies by Number of Tweets', fontsize=16, fontweight='bold')
plt.xlabel('Stock / Company', fontsize=12)
plt.ylabel('Number of Tweets', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(f"Most tweeted company: {stock_counts.index[0]} with {stock_counts.values[0]:,} tweets")

In [ ]:
# --- Date range analysis ---
date_column = None
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower() or 'created' in col.lower():
        date_column = col
        break

if date_column:
    df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
    
    # Drop NaT values before calculating range
    valid_dates = df[date_column].dropna()
    
    print("=" * 50)
    print(f"DATE RANGE ANALYSIS (column: '{date_column}')")
    print("=" * 50)
    
    if len(valid_dates) > 0:
        earliest = valid_dates.min()
        latest   = valid_dates.max()
        
        print(f"Earliest tweet : {earliest}")
        print(f"Latest tweet   : {latest}")
        
        # Fix: convert to Python native datetime before subtracting
        # This avoids the pandas OutOfBoundsDatetime overflow error
        try:
            delta = latest.to_pydatetime() - earliest.to_pydatetime()
            print(f"Date range     : {delta.days} days")
        except Exception:
            print(f"Date range     : Unable to calculate (dates may be out of bounds)")
        
        print(f"Valid date rows : {len(valid_dates):,} out of {len(df):,}")
    else:
        print("No valid dates found after parsing.")
else:
    print("No date column found in dataset.")

In [ ]:
# --- Number of tweets per date (if date column exists) ---
if date_column and df[date_column].notna().sum() > 0:
    df['date_only'] = df[date_column].dt.date
    tweets_per_date = df.groupby('date_only').size()

    plt.figure(figsize=(16, 5))
    tweets_per_date.plot(kind='line', color='steelblue', linewidth=1.5)
    plt.title('Number of Tweets Per Day Over Time', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Number of Tweets', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"Average tweets per day: {tweets_per_date.mean():.0f}")
    print(f"Peak tweets in a day  : {tweets_per_date.max():,} on {tweets_per_date.idxmax()}")
else:
    print("Skipping date trend chart — no valid date column found.")

## Text Exploratory Data Analysis

In [ ]:
# --- Drop rows with missing tweet text ---
print(f"Rows before dropping missing tweets: {len(df):,}")
df = df.dropna(subset=['TWEET'])
print(f"Rows after dropping missing tweets : {len(df):,}")

In [ ]:
# --- Tweet length distribution ---
df['tweet_length'] = df['TWEET'].astype(str).apply(len)
df['word_count']   = df['TWEET'].astype(str).apply(lambda x: len(x.split()))

print("=" * 50)
print("TWEET LENGTH STATISTICS")
print("=" * 50)
print(f"Average tweet length (chars) : {df['tweet_length'].mean():.1f}")
print(f"Average tweet word count     : {df['word_count'].mean():.1f}")
print(f"Shortest tweet               : {df['tweet_length'].min()} characters")
print(f"Longest tweet                : {df['tweet_length'].max()} characters")

In [ ]:
# --- Tweet length histogram ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['tweet_length'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Tweet Length (Characters)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Number of Tweets')

axes[1].hist(df['word_count'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Tweet Word Count', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Number of Tweets')

plt.tight_layout()
plt.show()

In [ ]:
# --- Top 15 most frequent words (basic word frequency) ---
all_tweets_text = ' '.join(df['TWEET'].astype(str)).lower()
words = re.findall(r'\b[a-z]{4,}\b', all_tweets_text)

# Remove common noise words that are not meaningful
noise_words = ['http', 'https', 'that', 'this', 'with', 'from', 'have', 'will',
               'your', 'they', 'been', 'were', 'said', 'what', 'when', 'their']
meaningful_words = [w for w in words if w not in noise_words]

word_counts = Counter(meaningful_words)
top_15 = word_counts.most_common(15)
top_words = [w[0] for w in top_15]
top_counts = [w[1] for w in top_15]

plt.figure(figsize=(14, 6))
bars = plt.bar(top_words, top_counts, color=sns.color_palette('Blues_r', 15))
plt.title('Top 15 Most Frequent Words in Tweets', fontsize=16, fontweight='bold')
plt.xlabel('Word', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, top_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 15 most frequent words (basic word frequency) ---
all_tweets_text = ' '.join(df['TWEET'].astype(str)).lower()
words = re.findall(r'\b[a-z]{4,}\b', all_tweets_text)

# Remove common noise words that are not meaningful
noise_words = ['http', 'https', 'that', 'this', 'with', 'from', 'have', 'will',
               'your', 'they', 'been', 'were', 'said', 'what', 'when', 'their']
meaningful_words = [w for w in words if w not in noise_words]

word_counts = Counter(meaningful_words)
top_15 = word_counts.most_common(15)
top_words = [w[0] for w in top_15]
top_counts = [w[1] for w in top_15]

plt.figure(figsize=(14, 6))
bars = plt.bar(top_words, top_counts, color=sns.color_palette('Blues_r', 15))
plt.title('Top 15 Most Frequent Words in Tweets', fontsize=16, fontweight='bold')
plt.xlabel('Word', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, top_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# Preprocessing


### Intsall Libraries & Download NLTK Resources

In [ ]:
# Install required libraries
!{sys.executable} -m pip install nltk wordcloud afinn scikit-learn seaborn textblob --quiet

import nltk

# Download all required NLTK resources
nltk.download('punkt')                      # Tokenization rules
nltk.download('punkt_tab')                  # Updated tokenization
nltk.download('stopwords')                  # Stopword list
nltk.download('wordnet')                    # Dictionary for lemmatization
nltk.download('omw-1.4')                    # Supporting data for wordnet
nltk.download('averaged_perceptron_tagger_eng')  # POS tagging
nltk.download('maxent_ne_chunker_tab')      # NER chunker
nltk.download('words')                      # Required for NER
nltk.download('vader_lexicon')              # VADER sentiment

print("\nAll resources downloaded successfully!")

## Data Cleaning

In [ ]:
import re

def clean_noise(text):
    """
    Removes URLs, @mentions, hashtags, special characters, and extra whitespace.
    Also applies case folding (lowercase).
    """
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # Remove URLs
    text = re.sub(r'@\w+', '', text)                         # Remove @mentions
    text = re.sub(r'#\w+', '', text)                         # Remove #hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)                  # Remove special chars & numbers
    text = text.lower()                                        # Case folding
    text = re.sub(r'\s+', ' ', text).strip()                 # Remove extra whitespace
    return text

# Apply to TWEET column and save in new column
df['clean_text'] = df['TWEET'].apply(clean_noise)

# Preview result
print("Before cleaning:")
print(df['TWEET'].iloc[0])
print("\nAfter cleaning:")
print(df['clean_text'].iloc[0])

## Sentence Segmentation

In [ ]:
from nltk.tokenize import sent_tokenize

# Apply sentence segmentation on the ORIGINAL tweet (before cleaning)
# We use raw text here because we want to preserve sentence boundaries
df['sentences'] = df['TWEET'].apply(sent_tokenize)

# Show example
print("Original tweet:")
print(df['TWEET'].iloc[0])
print("\nSentences:")
for i, s in enumerate(df['sentences'].iloc[0]):
    print(f"  Sentence {i+1}: {s}")

## Tokenization

In [ ]:
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    """
    Tokenizes text into words and removes punctuation.
    isalpha() keeps only words with letters (no numbers, no symbols).
    """
    tokens = word_tokenize(str(text))
    # Remove punctuation — keep only alphabetical tokens
    tokens = [word for word in tokens if word.isalpha()]
    return tokens

df['tokens'] = df['clean_text'].apply(tokenize_text)

# Preview
print("Cleaned text:")
print(df['clean_text'].iloc[0])
print("\nTokens:")
print(df['tokens'].iloc[0])

## Handling Abbreviations

In [ ]:
# Abbreviation dictionary — tailored for stock market / financial tweets
abbreviation_dict = {
    'rt'  : 'retweet',
    'roi' : 'return on investment',
    'ipo' : 'initial public offering',
    'etf' : 'exchange traded fund',
    'vix' : 'volatility index',
    'pe'  : 'price earnings',
    'eps' : 'earnings per share',
    'ytd' : 'year to date',
    'q'   : 'quarter',
    'u'   : 'you',
    'ur'  : 'your',
    'dm'  : 'direct message',
    'omg' : 'oh my god',
    'lol' : 'laugh out loud',
    'imo' : 'in my opinion',
    'ngl' : 'not going to lie',
    'tbh' : 'to be honest',
    'asap': 'as soon as possible',
    'fyi' : 'for your information',
}

def expand_abbreviations(token_list):
    """
    Replaces abbreviations in the token list with their full forms.
    Returns a flat list of tokens after expansion.
    """
    expanded = []
    for word in token_list:
        expanded_word = abbreviation_dict.get(word.lower(), word)
        expanded.extend(expanded_word.split())
    return expanded

df['expanded_tokens'] = df['tokens'].apply(expand_abbreviations)

# Preview
print("Tokens before abbreviation expansion:")
print(df['tokens'].iloc[0])
print("\nTokens after abbreviation expansion:")
print(df['expanded_tokens'].iloc[0])

## Stopword Removal

In [ ]:
from nltk.corpus import stopwords

# Load NLTK English stopwords
stop_words = set(stopwords.words('english'))

# Add custom stopwords relevant to the stock tweet dataset
custom_stopwords = [
    'next', 'via', 'amp', 'rt', 'co',
    'stock', 'market', 'share', 'price',   # too generic for this dataset
    'today', 'year', 'time', 'make', 'said'
]
stop_words.update(custom_stopwords)

def remove_stopwords(token_list):
    """
    Removes stopwords from the token list.
    Only keeps words that are NOT in the stopword set.
    """
    return [word for word in token_list if word not in stop_words]

df['filtered_tokens'] = df['expanded_tokens'].apply(remove_stopwords)

# Preview — compare before and after
print(f"Tokens before stopword removal ({len(df['expanded_tokens'].iloc[0])} words):")
print(df['expanded_tokens'].iloc[0])
print(f"\nTokens after stopword removal ({len(df['filtered_tokens'].iloc[0])} words):")
print(df['filtered_tokens'].iloc[0])

## Stemming & Lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def apply_stemming(token_list):
    """Applies Porter Stemmer — chops word endings using rules."""
    return [stemmer.stem(word) for word in token_list]

def apply_lemmatization(token_list):
    """Applies WordNet Lemmatizer — looks up dictionary root form."""
    return [lemmatizer.lemmatize(word) for word in token_list]

df['stemmed_words']    = df['filtered_tokens'].apply(apply_stemming)
df['lemmatized_words'] = df['filtered_tokens'].apply(apply_lemmatization)

print("Stemming and Lemmatization complete!")
print("\nExample comparison:")
print(f"Filtered tokens  : {df['filtered_tokens'].iloc[0]}")
print(f"Stemmed          : {df['stemmed_words'].iloc[0]}")
print(f"Lemmatized       : {df['lemmatized_words'].iloc[0]}")

In [ ]:
# --- Full pipeline comparison for one tweet ---
row = 5   # Change this to inspect different tweets

print("FULL PREPROCESSING PIPELINE COMPARISON")
print("=" * 60)
print(f"1. ORIGINAL TWEET:\n   {df['TWEET'].iloc[row]}")
print(f"\n2. CLEANED (noise removed, lowercased):\n   {df['clean_text'].iloc[row]}")
print(f"\n3. TOKENIZED (words split, punctuation removed):\n   {df['tokens'].iloc[row]}")
print(f"\n4. ABBREVIATIONS EXPANDED:\n   {df['expanded_tokens'].iloc[row]}")
print(f"\n5. STOPWORDS REMOVED:\n   {df['filtered_tokens'].iloc[row]}")
print(f"\n6a. STEMMED (chopped endings):\n   {df['stemmed_words'].iloc[row]}")
print(f"\n6b. LEMMATIZED (dictionary root — USE THIS):\n   {df['lemmatized_words'].iloc[row]}")

## Pos Tagging

In [ ]:
import nltk

def apply_pos_tagging(token_list):
    """Applies POS tagging to a list of tokens."""
    return nltk.pos_tag(token_list)

# Apply POS tagging to lemmatized words
df['pos_tags'] = df['lemmatized_words'].apply(apply_pos_tagging)

# Preview
print("Lemmatized tokens:")
print(df['lemmatized_words'].iloc[0])
print("\nPOS Tags:")
print(df['pos_tags'].iloc[0])
print("\nPOS Tag reference: NN=Noun, VB=Verb, JJ=Adjective, RB=Adverb, NNP=Proper Noun")

In [ ]:
# --- Most common POS tags in the dataset ---
all_tags = []
for tag_list in df['pos_tags']:
    all_tags.extend([tag for _, tag in tag_list])

tag_counts = Counter(all_tags).most_common(10)
tag_labels = [t[0] for t in tag_counts]
tag_values = [t[1] for t in tag_counts]

plt.figure(figsize=(12, 5))
sns.barplot(x=tag_labels, y=tag_values, palette='coolwarm')
plt.title('Top 10 Most Common POS Tags in Tweets', fontsize=14, fontweight='bold')
plt.xlabel('POS Tag')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Chunking

In [ ]:
# Define a grammar rule to extract Noun Phrases (NP)
# Rule: optional determiner + any adjectives + a noun
grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
chunk_parser = nltk.RegexpParser(grammar)

def extract_noun_phrases(pos_tagged_tokens):
    """Extracts noun phrases from POS-tagged tokens using chunking."""
    tree = chunk_parser.parse(pos_tagged_tokens)
    noun_phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == 'NP':
            phrase = ' '.join(word for word, tag in subtree.leaves())
            noun_phrases.append(phrase)
    return noun_phrases

df['noun_phrases'] = df['pos_tags'].apply(extract_noun_phrases)

# Preview
print("POS Tags:")
print(df['pos_tags'].iloc[0])
print("\nExtracted Noun Phrases:")
print(df['noun_phrases'].iloc[0])

In [ ]:
# --- Top 15 most common noun phrases across all tweets ---
all_noun_phrases = []
for phrase_list in df['noun_phrases']:
    all_noun_phrases.extend(phrase_list)

np_counts = Counter(all_noun_phrases).most_common(15)
np_words  = [n[0] for n in np_counts]
np_values = [n[1] for n in np_counts]

plt.figure(figsize=(14, 6))
sns.barplot(x=np_values, y=np_words, palette='viridis')
plt.title('Top 15 Most Common Noun Phrases in Stock Tweets', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.ylabel('Noun Phrase')
plt.tight_layout()
plt.show()

## Named Entity Recognition

In [ ]:
def apply_ner(raw_text):
    """
    Applies Named Entity Recognition to RAW text (not cleaned).
    NER needs original capitalisation to identify proper nouns.
    Returns a list of (entity_text, entity_type) tuples.
    """
    tokens  = nltk.word_tokenize(str(raw_text))
    pos_tag = nltk.pos_tag(tokens)
    tree    = nltk.ne_chunk(pos_tag)
    
    entities = []
    for chunk in tree:
        if hasattr(chunk, 'label'):
            entity_text = ' '.join(c[0] for c in chunk)
            entity_type = chunk.label()
            entities.append((entity_text, entity_type))
    return entities

# Apply to a sample (NER is slow — run on first 500 rows for demonstration)
print("Applying NER to sample of 500 tweets")
df_sample_ner = df.head(500).copy()
df_sample_ner['ner_entities'] = df_sample_ner['TWEET'].apply(apply_ner)

# Store back into main df for the sample rows
df.loc[df_sample_ner.index, 'ner_entities'] = df_sample_ner['ner_entities']

print("NER complete!")
print("\nExample:")
print(f"Tweet   : {df['TWEET'].iloc[0]}")
print(f"Entities: {df_sample_ner['ner_entities'].iloc[0]}")

In [ ]:
# --- Most frequently identified entities and their types ---
all_entities = []
for ent_list in df_sample_ner['ner_entities']:
    all_entities.extend(ent_list)

if all_entities:
    entity_df = pd.DataFrame(all_entities, columns=['Entity', 'Type'])
    
    print("=" * 50)
    print("TOP 15 MOST FREQUENT ENTITIES")
    print("=" * 50)
    print(entity_df['Entity'].value_counts().head(15))
    
    print("\n" + "=" * 50)
    print("ENTITY TYPE DISTRIBUTION")
    print("=" * 50)
    print(entity_df['Type'].value_counts())
    
    # Plot entity type distribution
    plt.figure(figsize=(8, 5))
    entity_df['Type'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title('Named Entity Types Found in Stock Tweets', fontsize=14, fontweight='bold')
    plt.xlabel('Entity Type')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No entities found in the sample.")

## Word Cloud Visualisation

In [ ]:
from wordcloud import WordCloud

# Join all lemmatized words into one big string
all_clean_words = []
for word_list in df['lemmatized_words']:
    all_clean_words.extend(word_list)

text_for_cloud = ' '.join(all_clean_words)

# Generate the word cloud
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    colormap='Blues',
    max_words=150,
    collocations=False   # Avoid repeating word pairs
).generate(text_for_cloud)

# Plot
plt.figure(figsize=(16, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Most Frequent Words in Stock Market Tweets',
          fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
print(f"Word cloud generated from {len(all_clean_words):,} cleaned tokens.")

In [ ]:
# --- Word cloud per company (top 3 stocks) ---
top_3_stocks = df['STOCK'].value_counts().head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, stock in zip(axes, top_3_stocks):
    stock_words = []
    for word_list in df[df['STOCK'] == stock]['lemmatized_words']:
        stock_words.extend(word_list)
    
    if stock_words:
        wc = WordCloud(width=600, height=400, background_color='white',
                       colormap='viridis', max_words=80).generate(' '.join(stock_words))
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f'Word Cloud — {stock}', fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Clouds by Top 3 Companies', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Feature Engineering

- TF-IDF Features
- Word Embedding Features
- BERT Embeddings

### TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join lemmatized tokens back into a string for TF-IDF
# TF-IDF requires a string input, not a list
df['joined_tokens'] = df['lemmatized_words'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else ''
)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,        # Keep top 1000 words only
    stop_words='english',     # Additional stopword removal
    ngram_range=(1, 2),       # Include single words AND 2-word phrases
)

# Fit and transform — learns vocabulary AND converts to numbers
tfidf_matrix = tfidf_vectorizer.fit_transform(df['joined_tokens'])

# Convert to readable DataFrame
feature_names = tfidf_vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=df.index
)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"(Rows = tweets, Columns = unique words/phrases)")
print(f"\nThis is called a SPARSE MATRIX because most values are 0.0")
print(f"Non-zero values: {tfidf_matrix.nnz:,} out of {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]:,} total")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.1f}% zeros")
df_tfidf.head()

In [ ]:
# --- Top TF-IDF terms overall ---
mean_tfidf = df_tfidf.mean().sort_values(ascending=False).head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=mean_tfidf.values, y=mean_tfidf.index, palette='Blues_r')
plt.title('Top 20 Words by Average TF-IDF Score', fontsize=14, fontweight='bold')
plt.xlabel('Mean TF-IDF Score')
plt.ylabel('Word / Phrase')
plt.tight_layout()
plt.show()
print("Higher score = more unique and important to specific tweets")

In [ ]:
# --- TF-IDF top terms per stock company ---
top_stocks_tfidf = df['STOCK'].value_counts().head(3).index.tolist()

print("=" * 60)
print("TOP 10 TF-IDF TERMS PER COMPANY")
print("=" * 60)

for stock in top_stocks_tfidf:
    stock_indices = df.index[df['STOCK'] == stock]
    if len(stock_indices) > 0:
        stock_tfidf_mean = df_tfidf.loc[stock_indices].mean().sort_values(ascending=False).head(10)
        print(f"\n{stock}:")
        for term, score in stock_tfidf_mean.items():
            print(f"  {term:<25} {score:.4f}")

### Word Embeddeding Features

In [ ]:
try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'gensim', '--quiet'])
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True

import numpy as np

# Prepare sentences — list of token lists
sentences = df['lemmatized_words'].tolist()
sentences = [s for s in sentences if isinstance(s, list) and len(s) > 0]

# Train Word2Vec model on our tweet corpus
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,    # 100-dimensional word vectors
    window=5,           # Context window size
    min_count=2,        # Ignore words appearing less than 2 times
    workers=4,          # Parallel threads
    epochs=10
)

print(f"Word2Vec model trained!")
print(f"Vocabulary size: {len(w2v_model.wv):,} words")
print(f"Vector dimensions: {w2v_model.vector_size}")

# Example: most similar words
try:
    print(f"\nWords most similar to 'stock':")
    for word, score in w2v_model.wv.most_similar('stock', topn=5):
        print(f"  {word:<20} similarity: {score:.3f}")
except KeyError:
    print("\n'stock' not in vocabulary — try another word")

In [ ]:
import numpy as np

def get_tweet_embedding(token_list, model, vector_size=100):
    """
    Converts a list of tokens into a single vector by averaging
    the word vectors of all known words in the tweet.
    """
    vectors = []
    for word in token_list:
        if word in model.wv:
            vectors.append(model.wv[word])
    
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)  # Return zero vector if no known words

# Generate embeddings for all tweets
embeddings = df['lemmatized_words'].apply(
    lambda x: get_tweet_embedding(x if isinstance(x, list) else [], w2v_model)
)

# Convert to DataFrame
embedding_cols = [f'w2v_{i}' for i in range(w2v_model.vector_size)]
df_embeddings = pd.DataFrame(embeddings.tolist(), columns=embedding_cols, index=df.index)

print(f"Word2Vec embeddings shape: {df_embeddings.shape}")
print(f"Each tweet is now represented as a {w2v_model.vector_size}-dimensional vector")
df_embeddings.head(3)

### BERT Embeddings

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentence-transformers', '--quiet'])
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True

# Load a lightweight pre-trained BERT model
print("Loading BERT model (this may take a moment on first run)...")
bert_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small but effective model
print("BERT model loaded!")

# Apply to a sample of tweets (BERT is slow on large datasets)
sample_size = min(500, len(df))
df_bert_sample = df.head(sample_size).copy()

print(f"\nGenerating BERT embeddings for {sample_size} tweets...")
bert_embeddings = bert_model.encode(
    df_bert_sample['TWEET'].astype(str).tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"\nBERT Embeddings shape: {bert_embeddings.shape}")
print(f"Each tweet is represented as a {bert_embeddings.shape[1]}-dimensional BERT vector")

In [ ]:
# --- Compare TF-IDF vs Word2Vec vs BERT ---
print("=" * 60)
print("FEATURE COMPARISON SUMMARY")
print("=" * 60)

comparison_data = {
    'Feature Type'   : ['TF-IDF', 'Word2Vec', 'BERT'],
    'Dimensions'     : [1000, 100, 384],
    'Type'           : ['Sparse (mostly zeros)', 'Dense (all values)', 'Dense (contextual)'],
    'Context Aware'  : ['No', 'Partial', 'Yes — fully contextual'],
    'Training'       : ['No training needed', 'Trained on our data', 'Pre-trained on massive corpus'],
    'Best For'       : ['Traditional ML models', 'Custom embeddings', 'Deep learning / transformers'],
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

### Preprocessing Summary

In [ ]:
print("=" * 60)
print("PREPROCESSING PIPELINE — COLUMNS CREATED")
print("=" * 60)
pipeline_columns = {
    'TWEET'             : 'Original raw tweet text',
    'clean_text'        : 'Noise removed + lowercased',
    'sentences'         : 'Split into individual sentences',
    'tokens'            : 'Tokenized + punctuation removed',
    'expanded_tokens'   : 'Abbreviations expanded',
    'filtered_tokens'   : 'Stopwords removed',
    'stemmed_words'     : 'Stemmed (Porter Stemmer)',
    'lemmatized_words'  : 'Lemmatized (WordNet) ← USE THIS',
    'pos_tags'          : 'Part-of-Speech tags assigned',
    'noun_phrases'      : 'Noun phrases extracted (chunking)',
    'ner_entities'      : 'Named entities identified',
    'joined_tokens'     : 'Lemmatized tokens joined as string',
}

for col, desc in pipeline_columns.items():
    status = '✓' if col in df.columns else '—'
    print(f"  {status}  {col:<22}  {desc}")

print(f"\nDataset ready for Sentiment Analysis!")
print(f"Total tweets: {len(df):,}")

# Sentiment Analysis

- TextBlob polarity
- VADER polarity
- Positive/Negative/Neutral labels

In [ ]:
# Sentiment target creation

sentiment_source_col = None
for candidate_col in ['LSTM_POLARITY', 'TEXTBLOB_POLARITY', 'vader_compound_score']:
    if candidate_col in df.columns:
        sentiment_source_col = candidate_col
        break

if sentiment_source_col is None:
    raise ValueError("No polarity score column found. Expected LSTM_POLARITY, TEXTBLOB_POLARITY, or vader_compound_score.")

# Clean numeric polarity values safely
sentiment_scores = pd.to_numeric(df[sentiment_source_col], errors='coerce')
if 'TEXTBLOB_POLARITY' in df.columns:
    sentiment_scores = sentiment_scores.fillna(pd.to_numeric(df['TEXTBLOB_POLARITY'], errors='coerce'))

def polarity_to_sentiment(score, threshold=0.05):
    if pd.isna(score):
        return np.nan
    if score > threshold:
        return 'positive'
    if score < -threshold:
        return 'negative'
    return 'neutral'

df['sentiment_score'] = sentiment_scores
df['sentiment_label'] = df['sentiment_score'].apply(polarity_to_sentiment)
df = df.dropna(subset=['TWEET', 'sentiment_label']).copy()

print(f"Sentiment source column: {sentiment_source_col}")
print(f"Rows with sentiment labels: {len(df):,}")
print(df[['TWEET', 'sentiment_score', 'sentiment_label']].head())


## Sentiment EDA 
   - Sentiment distribution
   - Sentiment by company
   - Sentiment over time
   - Sentiment vs stock return

In [ ]:
#  Sentiment EDA 
sentiment_order = ['negative', 'neutral', 'positive']
sentiment_colors = {
    'negative': 'red',
    'neutral': 'grey',
    'positive': 'green'
}
sentiment_counts = df['sentiment_label'].value_counts().reindex(sentiment_order).fillna(0).astype(int)

print("Sentiment distribution:")
print(sentiment_counts)
print("\nSentiment percentage:")
print((sentiment_counts / sentiment_counts.sum() * 100).round(2))

plt.figure(figsize=(7, 4))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette=[sentiment_colors[label] for label in sentiment_counts.index])
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Tweets')
plt.show()

if 'STOCK' in df.columns:
    stock_sentiment = pd.crosstab(df['STOCK'], df['sentiment_label'], normalize='index')
    top_stocks = df['STOCK'].value_counts().head(10).index
    stock_sentiment.loc[top_stocks, sentiment_order].plot(kind='bar', stacked=True, figsize=(12, 5), color=[sentiment_colors[label] for label in sentiment_order])
    plt.title('Sentiment Proportion by Top Stocks')
    plt.xlabel('Stock')
    plt.ylabel('Proportion')
    plt.legend(title='Sentiment')
    plt.tight_layout()
    plt.show()


# Opinion Mining

## Opinion Word Extraction

In [ ]:
#  Opinion Word Extraction 
# Opinion words are sentiment-bearing terms. We combine a finance-aware mini lexicon with VADER if available.

positive_lexicon = {
    'good', 'great', 'excellent', 'strong', 'growth', 'gain', 'gains', 'up', 'bullish',
    'profit', 'profits', 'beat', 'beats', 'buy', 'rally', 'surge', 'positive', 'improve'
}
negative_lexicon = {
    'bad', 'poor', 'weak', 'loss', 'losses', 'down', 'bearish', 'drop', 'fall', 'falls',
    'miss', 'misses', 'sell', 'crash', 'risk', 'negative', 'decline', 'lawsuit', 'debt'
}
opinion_lexicon = positive_lexicon.union(negative_lexicon)

try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
except Exception:
    vader = None

def extract_opinion_words(tokens):
    if not isinstance(tokens, list):
        return []
    extracted = []
    for token in tokens:
        token_lower = str(token).lower()
        if token_lower in opinion_lexicon:
            extracted.append(token_lower)
        elif vader is not None and abs(vader.polarity_scores(token_lower)['compound']) >= 0.3:
            extracted.append(token_lower)
    return sorted(set(extracted))

df['opinion_words'] = df['lemmatized_words'].apply(extract_opinion_words)

all_opinion_words = [word for words in df['opinion_words'] for word in words]
opinion_word_freq = Counter(all_opinion_words)
print("Top opinion words:")
print(pd.DataFrame(opinion_word_freq.most_common(20), columns=['opinion_word', 'frequency']))

df[['TWEET', 'sentiment_label', 'opinion_words']].head(10)


## Opinion Target Extraction

In [ ]:
# Opinion Target Extraction
# Opinion targets are the entities or noun phrases being discussed.
# Use a sample first because running spaCy over the full dataset can take a long time.

try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        nlp = spacy.blank('en')
        nlp.add_pipe('sentencizer')
    SPACY_AVAILABLE = True
except Exception:
    nlp = None
    SPACY_AVAILABLE = False

company_terms = set(df['STOCK'].dropna().astype(str).str.lower().unique()) if 'STOCK' in df.columns else set()

def extract_opinion_targets(text, max_targets=6):
    text = str(text)
    targets = []
    if SPACY_AVAILABLE and nlp is not None:
        doc = nlp(text[:1000])
        if hasattr(doc, 'noun_chunks'):
            try:
                targets.extend([chunk.text.lower().strip() for chunk in doc.noun_chunks])
            except Exception:
                pass
        targets.extend([ent.text.lower().strip() for ent in doc.ents])
    else:
        tokens = word_tokenize(text)
        tagged = nltk.pos_tag(tokens)
        targets.extend([word.lower() for word, tag in tagged if tag.startswith('NN')])

    for term in company_terms:
        if term and term in text.lower():
            targets.append(term)

    cleaned_targets = []
    for target in targets:
        target = re.sub(r'[^a-zA-Z\s]', '', target).strip()
        if 2 <= len(target) <= 60 and target not in stop_words:
            cleaned_targets.append(target)
    return list(dict.fromkeys(cleaned_targets))[:max_targets]

# Sample first for faster experimentation.
opinion_sample_size = min(5000, len(df))
opinion_sample_df = df.sample(opinion_sample_size, random_state=42).copy()

opinion_sample_df['opinion_targets'] = opinion_sample_df['TWEET'].apply(extract_opinion_targets)

# Optional: store sample results back into df for rows that were processed.
df['opinion_targets'] = None
df.loc[opinion_sample_df.index, 'opinion_targets'] = opinion_sample_df['opinion_targets']

target_freq = Counter([target for targets in opinion_sample_df['opinion_targets'] for target in targets])
print(f"Opinion target extraction completed on {opinion_sample_size:,} sampled tweets.")
print("Top opinion targets:")
print(pd.DataFrame(target_freq.most_common(20), columns=['target', 'frequency']))

opinion_sample_df[['TWEET', 'opinion_words', 'opinion_targets']].head(10)


## Dependancy Parsing

In [ ]:
# Dependency Parsing for Explicit Opinion-Target Pairs
# When a spaCy statistical model is available, dependencies link opinion words to their targets.
# If only a blank spaCy model is available, a nearby-word fallback is used.

def extract_dependency_opinions(text):
    text = str(text)
    pairs = []

    if SPACY_AVAILABLE and nlp is not None and 'parser' in nlp.pipe_names:
        doc = nlp(text[:1000])
        for token in doc:
            token_lower = token.lemma_.lower() if token.lemma_ else token.text.lower()
            if token_lower in opinion_lexicon or (vader is not None and abs(vader.polarity_scores(token.text)['compound']) >= 0.3):
                linked_targets = []
                if token.head.pos_ in ['NOUN', 'PROPN']:
                    linked_targets.append(token.head.text.lower())
                linked_targets.extend([child.text.lower() for child in token.children if child.pos_ in ['NOUN', 'PROPN']])
                for target in linked_targets:
                    pairs.append({'opinion': token.text.lower(), 'target': target, 'dependency': token.dep_})
    else:
        tokens = [token.lower() for token in word_tokenize(text) if token.isalpha()]
        for index, token in enumerate(tokens):
            if token in opinion_lexicon:
                window = tokens[max(0, index - 4): index] + tokens[index + 1: index + 5]
                candidate_targets = [word for word in window if word not in stop_words and word not in opinion_lexicon]
                for target in candidate_targets[:2]:
                    pairs.append({'opinion': token, 'target': target, 'dependency': 'window_context'})

    return pairs[:6]

# Use the same sample from Opinion Target Extraction
dependency_df = opinion_sample_df.copy() if 'opinion_sample_df' in globals() else df.sample(min(5000, len(df)), random_state=42).copy()

dependency_df['opinion_target_pairs'] = dependency_df['TWEET'].apply(extract_dependency_opinions)

pair_rows = []
for _, row in dependency_df.iterrows():
    for pair in row['opinion_target_pairs']:
        pair_rows.append(pair)

opinion_pairs_df = pd.DataFrame(pair_rows)

print(f"Dependency parsing completed on {len(dependency_df):,} sampled tweets.")
print("Extracted opinion-target pairs:")
display(
    opinion_pairs_df.head(20)
    if not opinion_pairs_df.empty
    else pd.DataFrame(columns=['opinion', 'target', 'dependency'])
)

# Aspect-Based Sentiment Analysis

## Aspect Extraction Using spaCy

In [ ]:
# Aspect Extraction Using spaCy
# ABSA starts by extracting aspect candidates such as product, service, battery, screen, price, or stock-related terms.
# Reuse the opinion-mining sample so this section stays fast.

absa_df = opinion_sample_df.copy() if 'opinion_sample_df' in globals() else df.sample(min(5000, len(df)), random_state=42).copy()

finance_aspect_terms = {
    'price', 'share', 'stock', 'market', 'earnings', 'revenue', 'profit', 'loss', 'growth',
    'forecast', 'sales', 'volume', 'volatility', 'return', 'dividend', 'product', 'service'
}

def extract_aspects(text, max_aspects=5):
    targets = extract_opinion_targets(text, max_targets=10)
    aspects = []
    for target in targets:
        if any(term in target.split() for term in finance_aspect_terms) or len(target.split()) <= 3:
            aspects.append(target)
    if not aspects:
        tokens = [token for token in tokenize_text(clean_noise(text)) if token not in stop_words]
        aspects = [token for token in tokens if token in finance_aspect_terms]
    return list(dict.fromkeys(aspects))[:max_aspects]

absa_df['aspects'] = absa_df['TWEET'].apply(extract_aspects)

aspect_freq = Counter([aspect for aspects in absa_df['aspects'] for aspect in aspects])
print(f"Aspect extraction completed on {len(absa_df):,} sampled tweets.")
print("Top extracted aspects:")
print(pd.DataFrame(aspect_freq.most_common(20), columns=['aspect', 'frequency']))

absa_df[['TWEET', 'aspects']].head(10)


## Aspect Categorization

In [ ]:
# Aspect Categorization
# Map noisy noun phrases into interpretable aspect categories.

aspect_category_keywords = {
    'Stock Performance': ['stock', 'share', 'price', 'return', 'market', 'rally', 'drop', 'volume'],
    'Financial Results': ['earnings', 'revenue', 'profit', 'loss', 'sales', 'dividend', 'quarter'],
    'Risk and Volatility': ['risk', 'volatility', 'debt', 'lawsuit', 'crash', 'decline'],
    'Product and Service': ['product', 'service', 'customer', 'quality', 'app', 'platform'],
    'Company News': ['company', 'ceo', 'deal', 'acquisition', 'launch', 'announcement']
}

def categorize_aspect(aspect):
    aspect_lower = str(aspect).lower()
    for category, keywords in aspect_category_keywords.items():
        if any(keyword in aspect_lower for keyword in keywords):
            return category
    return 'Other'

def categorize_aspects(aspects):
    return [{'aspect': aspect, 'category': categorize_aspect(aspect)} for aspect in aspects]

absa_df['aspect_categories'] = absa_df['aspects'].apply(categorize_aspects)

aspect_category_rows = []
for idx, row in absa_df.iterrows():
    for item in row['aspect_categories']:
        aspect_category_rows.append({
            'tweet_index': idx,
            'aspect': item['aspect'],
            'category': item['category'],
            'sentiment_label': row['sentiment_label']
        })

aspect_category_df = pd.DataFrame(aspect_category_rows)
print("Aspect category distribution:")
display(aspect_category_df['category'].value_counts().to_frame('count') if not aspect_category_df.empty else pd.DataFrame())


## Aspect Sentiment Classification

In [ ]:
#  Aspect Sentiment Classification 
# Each extracted aspect inherits context from the tweet and is assigned sentiment.
# A lightweight lexicon score is always available; an optional BERT-style transformer pipeline is used when installed.

try:
    from transformers import pipeline
    aspect_sentiment_pipeline = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
except Exception:
    aspect_sentiment_pipeline = None

def score_aspect_context(text, aspect):
    text = str(text)
    aspect = str(aspect)
    context = f"Aspect: {aspect}. Review: {text}"

    if vader is not None:
        score = vader.polarity_scores(context)['compound']
    else:
        opinion_words = extract_opinion_words(tokenize_text(clean_noise(context)))
        score = sum(1 for word in opinion_words if word in positive_lexicon) - sum(1 for word in opinion_words if word in negative_lexicon)
        score = score / max(len(opinion_words), 1)

    lexicon_label = polarity_to_sentiment(score, threshold=0.05)

    transformer_label = None
    transformer_score = None
    if aspect_sentiment_pipeline is not None:
        try:
            result = aspect_sentiment_pipeline(context[:512])[0]
            transformer_label = 'positive' if result['label'].upper().startswith('POS') else 'negative'
            transformer_score = result['score']
        except Exception:
            transformer_label = None
            transformer_score = None

    return {
        'aspect': aspect,
        'lexicon_score': score,
        'lexicon_sentiment': lexicon_label,
        'bert_absa_sentiment': transformer_label if transformer_label is not None else lexicon_label,
        'bert_absa_confidence': transformer_score
    }

def classify_aspect_sentiments(row):
    return [score_aspect_context(row['TWEET'], aspect) for aspect in row['aspects']]

absa_df['aspect_sentiments'] = absa_df.apply(classify_aspect_sentiments, axis=1)

aspect_sentiment_rows = []
for idx, row in absa_df.iterrows():
    for item in row['aspect_sentiments']:
        aspect_sentiment_rows.append({
            'tweet_index': idx,
            'aspect': item['aspect'],
            'category': categorize_aspect(item['aspect']),
            'sentiment': item['bert_absa_sentiment'],
            'lexicon_score': item['lexicon_score'],
            'confidence': item['bert_absa_confidence']
        })

aspect_sentiment_df = pd.DataFrame(aspect_sentiment_rows)
display(aspect_sentiment_df.head(20) if not aspect_sentiment_df.empty else pd.DataFrame())


## Aspect-Wise Sentiment Visualisation

In [ ]:
# Aspect-Wise Sentiment Visualisation
if not aspect_sentiment_df.empty:
    aspect_sentiment_summary = pd.crosstab(
        aspect_sentiment_df['category'],
        aspect_sentiment_df['sentiment']
    ).reindex(columns=sentiment_order, fill_value=0)

    display(aspect_sentiment_summary)

    aspect_sentiment_summary.plot(kind='bar', stacked=True, figsize=(11, 5), colormap='Set2')
    plt.title('Aspect-Based Sentiment by Category')
    plt.xlabel('Aspect Category')
    plt.ylabel('Number of Aspect Mentions')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No aspect sentiments were extracted. Check aspect extraction output above.")


# Modelling

- Feature selection
- Train-test split
- Model training
- Model evaluation

In [ ]:
#  Shared Modelling Setup 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# Use 50% of the labelled dataset for Traditional ML and Deep Learning models.
full_model_df = df[['TWEET', 'joined_tokens', 'sentiment_label']].dropna().copy()
model_df = full_model_df.sample(frac=0.5, random_state=42).reset_index(drop=True)

print(f"Total labelled rows available: {len(full_model_df):,}")
print(f"Rows used for ML/DL modelling (50% sample): {len(model_df):,}")
print("Class distribution used for modelling:")
print(model_df['sentiment_label'].value_counts())

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(model_df['sentiment_label'])
class_names = label_encoder.classes_

X_train_text, X_test_text, y_train, y_test = train_test_split(
    model_df['joined_tokens'].fillna('').astype(str),
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

raw_train_text = model_df.loc[X_train_text.index, 'TWEET'].astype(str)
raw_test_text = model_df.loc[X_test_text.index, 'TWEET'].astype(str)

model_results = []
model_predictions = {}
model_truths = {}

def evaluate_model(model_name, model_type, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    model_results.append({
        'Model': model_name,
        'Type': model_type,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })
    model_predictions[model_name] = y_pred
    model_truths[model_name] = y_true
    print(f"\n{model_name}")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

print(f"Training rows: {len(X_train_text):,}")
print(f"Testing rows : {len(X_test_text):,}")
print(f"Classes      : {list(class_names)}")


## Lexicon-Based Sentiment Model - VADER


In [ ]:
# --- Lexicon-Based Sentiment Model: VADER ---
# VADER is a rule/lexicon-based sentiment model that scores text using sentiment dictionaries and heuristics.
# It does not learn from the training data, so it provides a useful baseline against ML, DL, and transformers.

try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    vader_model = SentimentIntensityAnalyzer()
    LEXICON_MODEL_NAME = 'VADER Lexicon'

    def lexicon_predict_label(text):
        score = vader_model.polarity_scores(str(text))['compound']
        return polarity_to_sentiment(score, threshold=0.05)

except Exception:
    from textblob import TextBlob
    LEXICON_MODEL_NAME = 'TextBlob Lexicon'

    def lexicon_predict_label(text):
        score = TextBlob(str(text)).sentiment.polarity
        return polarity_to_sentiment(score, threshold=0.05)

lexicon_pred_labels = raw_test_text.apply(lexicon_predict_label)
lexicon_predictions = label_encoder.transform(lexicon_pred_labels)

evaluate_model(LEXICON_MODEL_NAME, 'Lexicon-Based', y_test, lexicon_predictions)


## Traditional ML Models - Logistic Regression, SVM, Random Forest


In [ ]:
# Traditional Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

traditional_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Linear SVM': LinearSVC(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1)
}

trained_traditional_models = {}
for model_name, classifier in traditional_models.items():
    pipeline_model = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
        ('classifier', classifier)
    ])
    pipeline_model.fit(X_train_text, y_train)
    predictions = pipeline_model.predict(X_test_text)
    trained_traditional_models[model_name] = pipeline_model
    evaluate_model(model_name, 'Traditional ML', y_test, predictions)


## Deep Learning Models - LSTM and BiLSTM


In [ ]:
# Deep Learning Models: LSTM and BiLSTM
try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except Exception as exc:
    TENSORFLOW_AVAILABLE = False
    print(f"TensorFlow is not available, skipping deep learning models: {exc}")

if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(42)
    max_words = 12000
    max_len = 60
    num_classes = len(class_names)

    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train_text)
    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=max_len, padding='post', truncating='post')
    X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_text), maxlen=max_len, padding='post', truncating='post')

    def build_lstm_model(bidirectional=False):
        model = Sequential()
        model.add(Embedding(input_dim=max_words, output_dim=64, input_length=max_len))
        if bidirectional:
            model.add(Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)))
        else:
            model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
        model.add(Dense(64, activation='relu'))
        model.add(Dropout(0.3))
        model.add(Dense(num_classes, activation='softmax'))
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        return model

    deep_learning_models = {
        'LSTM': build_lstm_model(bidirectional=False),
        'BiLSTM': build_lstm_model(bidirectional=True)
    }

    trained_deep_models = {}
    for model_name, dl_model in deep_learning_models.items():
        print(f"\nTraining {model_name}...")
        dl_model.fit(
            X_train_seq,
            y_train,
            validation_split=0.2,
            epochs=5,
            batch_size=64,
            callbacks=[EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)],
            verbose=1
        )
        probabilities = dl_model.predict(X_test_seq, verbose=0)
        predictions = np.argmax(probabilities, axis=1)
        trained_deep_models[model_name] = dl_model
        evaluate_model(model_name, 'Deep Learning', y_test, predictions)


## Transformer Models - BERT and RoBERTa


In [ ]:
# Transformer Models: BERT and RoBERTa
# Fine-tunes two transformer-family models on a smaller balanced sample to keep runtime practical.
# This version uses a small custom PyTorch Dataset, so it does not require the separate `datasets` package.
try:
    import torch
    from torch.utils.data import Dataset as TorchDataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    TRANSFORMERS_AVAILABLE = True
except Exception as exc:
    TRANSFORMERS_AVAILABLE = False
    print(f"Transformers stack is not available, skipping transformer models: {exc}")

class TweetSentimentDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.encodings = tokenizer(
            list(texts),
            padding='max_length',
            truncation=True,
            max_length=max_length
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

if TRANSFORMERS_AVAILABLE:
    transformer_sample_size = min(1200, len(model_df))
    transformer_df = (
        model_df.groupby('sentiment_label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), max(1, transformer_sample_size // model_df['sentiment_label'].nunique())), random_state=42))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    transformer_df['label'] = label_encoder.transform(transformer_df['sentiment_label'])

    tr_train, tr_test = train_test_split(
        transformer_df[['TWEET', 'label']],
        test_size=0.2,
        random_state=42,
        stratify=transformer_df['label']
    )

    transformer_specs = {
        'BERT': 'distilbert-base-uncased',
        'RoBERTa': 'distilroberta-base'
    }

    trained_transformer_models = {}
    for model_name, checkpoint in transformer_specs.items():
        print(f"\nFine-tuning {model_name} ({checkpoint})...")
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            num_labels=len(class_names),
            id2label={i: label for i, label in enumerate(class_names)},
            label2id={label: i for i, label in enumerate(class_names)}
        )

        train_dataset = TweetSentimentDataset(
            tr_train['TWEET'].astype(str).tolist(),
            tr_train['label'].tolist(),
            tokenizer,
            max_length=96
        )
        test_dataset = TweetSentimentDataset(
            tr_test['TWEET'].astype(str).tolist(),
            tr_test['label'].tolist(),
            tokenizer,
            max_length=96
        )

        training_args = TrainingArguments(
            output_dir=f'./{model_name.lower()}_sentiment_results',
            num_train_epochs=1,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=16,
            learning_rate=2e-5,
            weight_decay=0.01,
            logging_steps=50,
            save_strategy='no',
            report_to='none'
        )

        trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=test_dataset)
        trainer.train()
        output = trainer.predict(test_dataset)
        predictions = np.argmax(output.predictions, axis=1)

        trained_transformer_models[model_name] = {'trainer': trainer, 'tokenizer': tokenizer, 'checkpoint': checkpoint}
        evaluate_model(model_name, 'Transformer', tr_test['label'].to_numpy(), predictions)


# Model Evaluation

## Accuracy

In [ ]:
# --- Accuracy Summary ---
model_results_df = pd.DataFrame(model_results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
if model_results_df.empty:
    print("No model results available yet. Run the modelling cells first.")
else:
    display(model_results_df[['Model', 'Type', 'Accuracy']].sort_values('Accuracy', ascending=False))


## Precision

## Recall

## F1-Score

## Confusion Matrix

# Model Comparison


# Stock Price Impact Analysis

# Conclusion